In [1]:
import os
import json
import pandas as pd

from pathlib import Path


# Get GEO_ID from folder name (same pattern as prepare_data.ipynb)
GEO_ID = os.path.basename(os.path.dirname(os.path.abspath('__file__')))


description_path = Path('description.json')
preprocessed_csv_path = Path(f"{GEO_ID}_preprocessed.csv")

if not description_path.exists():
    raise FileNotFoundError(
        "description.json not found. Please run 'prepare_data.ipynb' first to generate it."
    )

if not preprocessed_csv_path.exists():
    raise FileNotFoundError(
        f"{preprocessed_csv_path} not found. Please run 'preprocess_data.ipynb' first to generate it."
    )

with open(description_path) as f:
    summary_data = json.load(f)

df = pd.read_csv(preprocessed_csv_path)

print("Data contains", df.shape[0], "samples and", df.shape[1], "columns.")
print("First few rows of the data:")
df.head()


Data contains 47 samples and 12497 columns.
First few rows of the data:


,efstime,efscens,stime,scens,sample_title,tissue,age at_diag,age at_first_event_or_censor,ethnicity,SAMD11,...,COX2,ATP8,ATP6,COX3,ND3,ND4L,ND4,ND5,ND6,CYTB
0,4043,0,4043,0,Neuroblastoma_usi_PALUKI,Neuroblastoma,1110,5152,Not Hispanic or Latino,1.747648,...,7.059210,4.891928,6.500045,8.330441,5.795434,5.946433,7.427408,6.659727,4.882422,6.943524
1,245,0,245,0,Neuroblastoma_usi_PALWKV,Neuroblastoma,2019,2263,Not Hispanic or Latino,2.440854,...,6.377251,4.579810,6.232081,7.965482,5.318799,5.543398,7.070861,6.233269,4.754535,6.328568
2,206,1,206,1,Neuroblastoma_usi_PAMKGY,Neuroblastoma,926,1131,Not Hispanic or Latino,2.622430,...,5.618590,3.651019,5.525917,7.109841,4.350829,4.528510,6.063517,5.487258,3.774227,5.495584
3,2466,0,2466,0,Neuroblastoma_usi_PAMMGU,Neuroblastoma,953,3418,Hispanic or Latino,1.796477,...,6.366620,4.494385,6.178376,7.621467,5.154473,5.125012,6.773641,6.126936,4.257590,6.334392
4,3838,0,3838,0,Neuroblastoma_usi_PANLCE,Neuroblastoma,1960,5797,Not Hispanic or Latino,2.633410,...,5.879444,4.100270,5.663046,6.738357,4.862939,5.308358,6.844380,6.236861,4.448032,5.939340


In [2]:
n_stats_written = 0

for idx, endpoint in enumerate(summary_data['survival-endpoints']):
    time_var = endpoint['time_var']['var_name']
    event_var = endpoint['event_var']['var_name']

    if time_var not in df.columns or event_var not in df.columns:
        abbrv = endpoint.get('abbrv', f'endpoint[{idx}]')
        print(
            f"Skipping stats for {abbrv}: columns not in preprocessed table "
            f"({time_var!r}, {event_var!r})."
        )
        endpoint.pop('stats', None)
        continue

    complete_mask = df[[time_var, event_var]].notnull().all(axis=1)
    n_complete = int(complete_mask.sum())
    n_incomplete = int(len(df) - n_complete)

    # Among samples with complete time + event data
    censored_count = int((complete_mask & (df[event_var] == 0.0)).sum())
    event_count = int((complete_mask & (df[event_var] == 1.0)).sum())

    censored_ratio = censored_count / n_complete

    print(f"Number of samples with incomplete data for {time_var} or {event_var}: {n_incomplete}")
    print(f"Number of samples with complete data for both variables: {n_complete}")
    print(f"Number of censored samples (event_var == 0.0): {censored_count}")
    print(f"Number of events (event_var == 1.0): {event_count}")
    pct = 100.0 * censored_ratio
    print(
        f"Censored ratio (among complete): {censored_count} / {n_complete} = "
        f"{censored_ratio:.3f} ({pct:.1f}%)"
    )

    # Refresh only this block on each run; leave all other endpoint fields unchanged
    endpoint['stats'] = {
        'n_incomplete': n_incomplete,
        'n_complete': n_complete,
        'n_censored': censored_count,
        'n_events': event_count,
        'censored_ratio': round(censored_ratio, 2),
    }
    n_stats_written += 1

with open(description_path, 'w', encoding='utf-8') as f:
    json.dump(summary_data, f, indent=4, ensure_ascii=False)
    f.write('\n')

print(
    f"Updated stats for {n_stats_written} of {len(summary_data['survival-endpoints'])} endpoint(s); "
    f"wrote {description_path.resolve()}"
)


Number of samples with incomplete data for efstime or efscens: 0
Number of samples with complete data for both variables: 47
Number of censored samples (event_var == 0.0): 37
Number of events (event_var == 1.0): 10
Censored ratio (among complete): 37 / 47 = 0.787 (78.7%)
Number of samples with incomplete data for stime or scens: 0
Number of samples with complete data for both variables: 47
Number of censored samples (event_var == 0.0): 41
Number of events (event_var == 1.0): 6
Censored ratio (among complete): 41 / 47 = 0.872 (87.2%)
Updated stats for 2 of 2 endpoint(s); wrote /Users/jakakokosar/dev/phd/survival-rnaseq-data/datasets/geo/GSE218527/description.json
